# SNAP Edge Case Analysis

Deep dive into the policy boundaries that trip up LLMs:
- Gross vs. net income thresholds
- Deduction stacking (earned income + standard + shelter)
- Elderly/disabled gross income waiver
- BBCE states vs. strict asset-test states
- Categorical eligibility overrides

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth.sources.us.snap import SNAPSource, BBCE_STATES, STRICT_ASSET_TEST_STATES
from govsynth.profiles.us_household import USHouseholdProfile
from govsynth.generators.snap_eligibility import SNAPEligibilityGenerator

print('Ready.')

Ready.


## 1. Gross Income Boundary — 3-Person Household, Virginia

In [2]:
source = SNAPSource(fiscal_year=2026, state='VA')
t = source.thresholds()
limits_3 = t.by_household_size(3)

print(f'3-person gross limit (130% FPL): ${limits_3.gross_monthly:,.2f}/month')
print(f'3-person net limit  (100% FPL): ${limits_3.net_monthly:,.2f}/month')
print()

offsets = [('5% below', -0.05), ('1% below', -0.01), ('AT LIMIT', 0.0), ('1% above', 0.01), ('5% above', 0.05)]
print(f'{"Label":<12} {"Gross Income":<18} {"Eligible?"}')
print('-' * 45)
for label, pct in offsets:
    profile = USHouseholdProfile.at_threshold(
        program='snap', threshold='gross_income_limit',
        state='VA', household_size=3, fiscal_year=2026,
        offset_pct=pct, seed=42,
    )
    eligible, reason = source.is_eligible(
        household_size=profile.household_size,
        gross_income=profile.monthly_gross_income,
        net_income=profile.monthly_net_income,
        liquid_assets=profile.liquid_assets,
    )
    print(f'{label:<12} ${profile.monthly_gross_income:>10,.2f}       {"ELIGIBLE" if eligible else "INELIGIBLE"}')


3-person gross limit (130% FPL): $2,888.00/month
3-person net limit  (100% FPL): $2,221.00/month

Label        Gross Income       Eligible?
---------------------------------------------
5% below     $  2,743.60       ELIGIBLE
1% below     $  2,859.12       ELIGIBLE
AT LIMIT     $  2,888.00       ELIGIBLE
1% above     $  2,916.88       INELIGIBLE
5% above     $  3,032.40       INELIGIBLE


## 2. Deduction Stacking — Net Income Edge Cases

The net income test is what trips models up. It requires applying earned income deduction, standard deduction, and optionally a shelter deduction. Let's show the mechanics for a household right at the net income limit.

In [ ]:
from govsynth.sources.us.snap import get_standard_deduction

hh_size = 4
limits_4 = t.by_household_size(4)
std_ded = get_standard_deduction(hh_size)

# Profile exactly at the net income limit
profile = USHouseholdProfile.at_threshold(
    program='snap', threshold='net_income_limit',
    state='VA', household_size=hh_size, fiscal_year=2026,
    offset_pct=0.0, seed=42,
)

earned_ded = (profile.earned_income or profile.monthly_gross_income) * 0.20
net_manual = profile.monthly_gross_income - earned_ded - std_ded
net_official = source.calculate_net_income(
    gross_income=profile.monthly_gross_income,
    household_size=hh_size,
    earned_income=profile.earned_income,
)

print(f'4-person net income limit: ${limits_4.net_monthly:,.2f}/month')
print()
print(f'Gross income:              ${profile.monthly_gross_income:>10,.2f}')
print(f'- 20% earned deduction:    ${earned_ded:>10,.2f}   (7 CFR 273.9(c)(1))')
print(f'- Standard deduction:      ${std_ded:>10,.2f}   (7 CFR 273.9(c)(2))')
print(f'= Net income:              ${net_manual:>10,.2f}')
print()
print(f'Official net (source):     ${net_official:>10,.2f}')
print(f'Limit:                     ${limits_4.net_monthly:>10,.2f}')
print(f'Eligible:                  {"YES" if net_official <= limits_4.net_monthly else "NO"}')


## 3. Elderly/Disabled Household — Gross Income Waiver

7 CFR 273.9(a)(1): Households with an elderly (60+) or disabled member are **exempt from the gross income test** but must still pass the net income test. This is a common LLM failure mode.

In [ ]:
# Household with income 20% ABOVE gross limit — would fail for normal household
# but is elderly, so gross test is waived
hh_size = 2
limits_2 = t.by_household_size(2)
gross_above = round(limits_2.gross_monthly * 1.20, 2)
net_below = round(limits_2.net_monthly * 0.80, 2)

# Normal household — ineligible
elig_normal, reason_normal = source.is_eligible(
    household_size=hh_size,
    gross_income=gross_above,
    net_income=net_below,
    liquid_assets=500,
    has_elderly_or_disabled=False,
)

# Elderly household — eligible (gross waived, net passes)
elig_elderly, reason_elderly = source.is_eligible(
    household_size=hh_size,
    gross_income=gross_above,
    net_income=net_below,
    liquid_assets=500,
    has_elderly_or_disabled=True,
)

print(f'2-person gross limit: ${limits_2.gross_monthly:,.2f} | net limit: ${limits_2.net_monthly:,.2f}')
print(f'Test household gross: ${gross_above:,.2f} (20% above limit) | net: ${net_below:,.2f}')
print()
print(f'Without elderly/disabled flag: {"ELIGIBLE" if elig_normal else "INELIGIBLE"}')
print(f'  Reason: {reason_normal}')
print()
print(f'With elderly/disabled flag:    {"ELIGIBLE" if elig_elderly else "INELIGIBLE"}')
print(f'  Reason: {reason_elderly}')
print()
print('Note: asset limit for elderly/disabled is $4,500 (not $3,000) per 7 CFR 273.8(b)(2)')


## 4. BBCE vs. Strict Asset Test — Virginia vs. Texas

Broad-Based Categorical Eligibility (BBCE) states waive the asset test for most households. Texas uses the strict federal asset test ($3,000 general). Same income, different outcomes based purely on state.

In [ ]:
source_va = SNAPSource(fiscal_year=2026, state='VA')  # BBCE
source_tx = SNAPSource(fiscal_year=2026, state='TX')  # Strict asset test

for state, src in [('VA (BBCE)', source_va), ('TX (strict)', source_tx)]:
    t_state = src.thresholds()
    asset_display = 'waived (BBCE)' if t_state.asset_limit_general is None else f'${t_state.asset_limit_general:,}'
    print(f'{state}: asset_limit_general = {asset_display}')

print()
# Household with $4,000 in assets, income well below limit
hh_size = 3
limits_3_va = source_va.thresholds().by_household_size(3)
gross = round(limits_3_va.gross_monthly * 0.70, 2)
assets = 4000

print(f'Test case: {hh_size}-person HH, gross=${gross:,.2f}, assets=${assets:,}')
print()
for state, src in [('VA (BBCE)', source_va), ('TX (strict)', source_tx)]:
    elig, reason = src.is_eligible(
        household_size=hh_size,
        gross_income=gross,
        liquid_assets=assets,
    )
    print(f'  {state}: {"ELIGIBLE" if elig else "INELIGIBLE"} — {reason[:80]}')


## 5. Generate Edge-Saturated Cases and Inspect Difficulty Distribution

In [ ]:
from collections import Counter

# Generate a larger batch to see difficulty spread
generator = SNAPEligibilityGenerator(fiscal_year=2026, state='VA')
cases = generator.generate(n=50, seed=42)

print(f'Generated: {len(cases)} cases')
print()

diff = Counter(c.difficulty.value for c in cases)
print('Difficulty distribution:')
for d, n in sorted(diff.items()):
    pct = n / len(cases) * 100
    bar = '█' * int(pct / 2)
    print(f'  {d:<12} {n:>3}  {pct:>5.1f}%  {bar}')

print()
print('Threshold types in cases:')
types = Counter(c.metadata.get('profile_strategy', 'unknown') for c in cases)
for t_type, n in sorted(types.items()):
    print(f'  {t_type:<30} {n}')


## 6. Spot Check: Inspect a Hard Ineligible Case's Rationale Trace

In [ ]:
hard_inelig = [c for c in cases if c.difficulty.value == 'hard' and c.expected_outcome == 'ineligible']
if hard_inelig:
    case = hard_inelig[0]
    print(f'Case: {case.case_id}')
    print(f'Outcome: {case.expected_outcome.upper()}')
    print(f'Scenario: {case.scenario.summary}')
    print()
    print('Rationale trace:')
    print(case.rationale_trace.to_plain_text())
else:
    print('No hard ineligible cases in this batch — try a larger n or different seed.')
